# NyayaRAG Kaggle Artifact Builder

Run this notebook on a Kaggle GPU (T4). It builds `nyayarag_artifacts.zip` and saves it to the notebook's **Output** tab for download. Unzip it locally to run the Streamlit app.

**Requirements:**
- Turn on **Internet** in notebook settings
- Turn on a **GPU** (T4 recommended)
- Add a Kaggle secret named `HF_TOKEN` with your Hugging Face token

In [ ]:
!nvidia-smi

In [ ]:
%cd /kaggle/working
!rm -rf NyayaRAG
!git clone https://github.com/mi-bilal/NyayaRAG.git
%cd /kaggle/working/NyayaRAG

## Configure HF Token

The build needs a Hugging Face token to download the corpus. Add `HF_TOKEN` in **Kaggle Secrets** (sidebar → Add-ons → Secrets) then run this cell.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf = secrets.get_secret("HF_TOKEN")

env = f"""HF_TOKEN={hf}
GROQ_MODEL=openai/gpt-oss-120b
EMBEDDING_MODEL=Qwen/Qwen3-Embedding-0.6B
EMBEDDING_DIM=1024
QDRANT_PATH=artifacts/qdrant
QDRANT_COLLECTION=nyayarag_statutes
BM25_PATH=artifacts/bm25/statutes_bm25.pkl
DATA_DIR=data
ARTIFACT_DIR=artifacts
TOP_K=5
CANDIDATE_POOL=80
RRF_K=60
MAX_CASE_TOKENS=3500
MAX_CONTEXT_TOKENS=5000
"""
open(".env", "w").write(env)
print("Wrote .env; HF token present:", bool(hf))

In [ ]:
!pip install . -q 2>&1 | tail -3

## Small Test Build

Run this first to prove everything works. It builds artifacts for 50 records only.

In [ ]:
!rm -rf artifacts
!python scripts/colab_full_build.py --batch-size 16 --upsert-batch-size 512 --embed-max-tokens 384 --embed-overlap-tokens 48 --log-every 1 --limit 50 --cache-dir /tmp/nyayarag_test_cache

## Full Build

Run this after the small test succeeds. Takes 10–30 minutes depending on GPU. The zip is saved to `/kaggle/working/` and appears in the **Output** tab when the notebook finishes.

In [ ]:
!rm -rf artifacts
!python scripts/colab_full_build.py --batch-size 64 --upsert-batch-size 1024 --embed-max-tokens 512 --embed-overlap-tokens 64 --log-every 10 --cache-dir /tmp/nyayarag_cache

In [ ]:
import shutil
from pathlib import Path

src = Path("artifacts/nyayarag_artifacts.zip")
dst = Path("/kaggle/working/nyayarag_artifacts.zip")
if src.exists():
    shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst} ({dst.stat().st_size / 1e6:.1f} MB)")
    print("\nDownload from: notebook Outputs tab -> nyayarag_artifacts.zip")
else:
    print(f"ERROR: {src} not found")